## Imports

In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

from neuro_fuzzy_toolbox import h_ANFIS, Hybrid_learning_algorithm, EarlyStopping, get_measures

In [2]:
import numpy as np

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Multiclass Classification

## Data

In [4]:
from ucimlrepo import fetch_ucirepo

In [5]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [6]:
X = X.fillna(value=0)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [8]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[115  38  25  25   9]


In [9]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[49 17 11 10  4]


In [10]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [11]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [12]:
y_train.dtype

torch.int64

In [13]:
x_train = x_train.type(torch.float32)
x_test = x_test.type(torch.float32)

In [14]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 4, shuffle = True)
x_trainset = loader.dataset.tensors[0]
y_trainset = loader.dataset.tensors[1]

In [15]:
y_trainset.dtype

torch.int64

## Model & Training

In [16]:
model = h_ANFIS(
    input_size = 13,
    num_mfs = 50,
    outputs = 5,
    rule_reduced = True,
    output_type='multiclass',
    dtype=torch.float32
)

In [17]:
loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.001, 'weight_decay': 0.01}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = EarlyStopping(
    patience=50, 
    delta=0.01
)

In [18]:
trainer = Hybrid_learning_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    validation=0.3,
    early_stopping=early_stopping
)

In [19]:
trainer(model, loader, verbose=True)

Epoch:   1/500 - loss: 1.349221 - validation loss: 1.328397
Epoch:   2/500 - loss: 1.304879 - validation loss: 1.328311
Epoch:   3/500 - loss: 1.347225 - validation loss: 1.365355
Epoch:   4/500 - loss: 1.280264 - validation loss: 1.300059
Epoch:   5/500 - loss: 1.331527 - validation loss: 1.341317
Epoch:   6/500 - loss: 1.263852 - validation loss: 1.328873
Epoch:   7/500 - loss: 1.279591 - validation loss: 1.356093
Epoch:   8/500 - loss: 1.346661 - validation loss: 1.380114
Epoch:   9/500 - loss: 1.314442 - validation loss: 1.415949
Epoch:  10/500 - loss: 1.324130 - validation loss: 1.419019
Epoch:  11/500 - loss: 1.485328 - validation loss: 1.480018
Epoch:  12/500 - loss: 1.321772 - validation loss: 1.427368
Epoch:  13/500 - loss: 1.319600 - validation loss: 1.417739
Epoch:  14/500 - loss: 1.397778 - validation loss: 1.438341
Epoch:  15/500 - loss: 1.301072 - validation loss: 1.419697
Epoch:  16/500 - loss: 1.316232 - validation loss: 1.412885
Epoch:  17/500 - loss: 1.309689 - valida

In [20]:
train_measures = get_measures(model, x_trainset, y_trainset)
for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.5613207547169812
Precision: 0.5307166646515318
Recall: 0.5613207547169812
F1: 0.5384540807495656
Confusion Matrix: [[96 16  1  1  1]
 [18  6  5  9  0]
 [ 8  3 10  4  0]
 [ 8  4  7  6  0]
 [ 1  1  3  3  1]]


In [21]:
test_measures = get_measures(model, x_test, y_test)
for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.5934065934065934
Precision: 0.5453303866928377
Recall: 0.5934065934065934
F1: 0.5582977776766597
Confusion Matrix: [[44  2  0  2  1]
 [10  3  2  2  0]
 [ 3  1  4  3  0]
 [ 1  2  4  3  0]
 [ 1  0  0  3  0]]
